# SVM urgency classification training

This notebook trains and evaluates three SVM-based urgency-classification methods using the prepared dataset splits:
1. Multiclass urgency classification (`Low`, `Medium`, `High`, `Critical`)
2. Cost-sensitive multiclass urgency classification
3. Binary screening for `High`/`Critical` vs routine review

The notebook prints classification reports and confusion matrices. It saves only the trained binary screening model and metadata under `train/artifacts`.


In [22]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_class_weight

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
PROJECT_ROOT = None

for candidate_root in candidate_roots:
    if (candidate_root / "data" / "processed").exists():
        PROJECT_ROOT = candidate_root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate project root from working directory: {cwd}"
    )

TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "incidents_train.csv"
VALIDATION_PATH = PROJECT_ROOT / "data" / "processed" / "incidents_validation.csv"
TEST_PATH = PROJECT_ROOT / "data" / "processed" / "incidents_test.csv"
ARTIFACTS_DIR = PROJECT_ROOT / "train" / "artifacts"

SCREENING_ARTIFACT_DIR = ARTIFACTS_DIR / "svm_urgency_binary_screening"
SCREENING_MODEL_PATH = SCREENING_ARTIFACT_DIR / "model.pkl"
SCREENING_METADATA_PATH = SCREENING_ARTIFACT_DIR / "metadata.json"

RANDOM_SEED = 42
TARGET_RECALL = 0.95

SCREENING_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Train path: {TRAIN_PATH}")
print(f"Validation path: {VALIDATION_PATH}")
print(f"Test path: {TEST_PATH}")


Project root: /home/lakshan/ssp/IMS
Train path: /home/lakshan/ssp/IMS/data/processed/incidents_train.csv
Validation path: /home/lakshan/ssp/IMS/data/processed/incidents_validation.csv
Test path: /home/lakshan/ssp/IMS/data/processed/incidents_test.csv


In [23]:
required_columns = {"feature_concatanation", "urgency", "urgency_label"}


def load_split(path):
    split_df = pd.read_csv(path)
    missing_columns = required_columns - set(split_df.columns)
    if missing_columns:
        raise ValueError(f"Dataset {path} is missing required columns: {sorted(missing_columns)}")

    split_df = split_df[["feature_concatanation", "urgency", "urgency_label"]].dropna().copy()
    split_df["feature_concatanation"] = split_df["feature_concatanation"].astype(str).str.strip()
    split_df = split_df[split_df["feature_concatanation"] != ""]
    return split_df


train_df = load_split(TRAIN_PATH)
validation_df = load_split(VALIDATION_PATH)
test_df = load_split(TEST_PATH)

label_mapping_df = (
    pd.concat(
        [
            train_df[["urgency", "urgency_label"]],
            validation_df[["urgency", "urgency_label"]],
            test_df[["urgency", "urgency_label"]],
        ],
        ignore_index=True,
    )
    .drop_duplicates()
    .sort_values("urgency_label")
    .reset_index(drop=True)
)
label_mapping = dict(zip(label_mapping_df["urgency_label"], label_mapping_df["urgency"]))

print(f"Train rows: {len(train_df)}")
print(f"Validation rows: {len(validation_df)}")
print(f"Test rows: {len(test_df)}")
label_mapping_df


Train rows: 2520
Validation rows: 540
Test rows: 540


,urgency,urgency_label
0,Low,0
1,Medium,1
2,High,2
3,Critical,3


In [24]:
def build_text_svm_pipeline(classifier):
    return Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=(1, 2),
                    min_df=2,
                    max_features=20000,
                    sublinear_tf=True,
                ),
            ),
            ("classifier", classifier),
        ]
    )


def print_multiclass_evaluation(title, y_true, y_pred, label_mapping):
    labels = sorted(label_mapping)
    print(title)
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Balanced accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}")
    print(f"Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")
    print("\nClassification report:\n")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=labels,
            target_names=[label_mapping[label] for label in labels],
            zero_division=0,
        )
    )
    confusion_df = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=labels),
        index=[f"actual_{label_mapping[label]}" for label in labels],
        columns=[f"pred_{label_mapping[label]}" for label in labels],
    )
    display(confusion_df)


def print_binary_evaluation(title, y_true, probabilities, threshold, label_mapping):
    predictions = (probabilities >= threshold).astype(int)
    print(title)
    print(f"Threshold: {threshold:.2f}")
    print(f"Accuracy: {accuracy_score(y_true, predictions):.4f}")
    print(f"Precision ({label_mapping[1]}): {precision_score(y_true, predictions, zero_division=0):.4f}")
    print(f"Recall ({label_mapping[1]}): {recall_score(y_true, predictions, zero_division=0):.4f}")
    print(f"F1 ({label_mapping[1]}): {f1_score(y_true, predictions, zero_division=0):.4f}")
    print(f"PR-AUC: {average_precision_score(y_true, probabilities):.4f}")
    print("\nClassification report:\n")
    print(
        classification_report(
            y_true,
            predictions,
            labels=[0, 1],
            target_names=[label_mapping[0], label_mapping[1]],
            zero_division=0,
        )
    )
    confusion_df = pd.DataFrame(
        confusion_matrix(y_true, predictions, labels=[0, 1]),
        index=[f"actual_{label_mapping[0]}", f"actual_{label_mapping[1]}"],
        columns=[f"pred_{label_mapping[0]}", f"pred_{label_mapping[1]}"],
    )
    display(confusion_df)


## 1. Multiclass SVM classifier

In [25]:
X_train = train_df["feature_concatanation"]
y_train = train_df["urgency_label"].astype(int)
X_validation = validation_df["feature_concatanation"]
y_validation = validation_df["urgency_label"].astype(int)
X_test = test_df["feature_concatanation"]
y_test = test_df["urgency_label"].astype(int)

baseline_urgency_classifier = build_text_svm_pipeline(
    LinearSVC(C=1.0, random_state=RANDOM_SEED, dual="auto")
)
baseline_urgency_classifier.fit(X_train, y_train)

baseline_validation_predictions = baseline_urgency_classifier.predict(X_validation)
baseline_test_predictions = baseline_urgency_classifier.predict(X_test)

print_multiclass_evaluation(
    "Validation results - baseline multiclass SVM",
    y_validation,
    baseline_validation_predictions,
    label_mapping,
)

print_multiclass_evaluation(
    "Test results - baseline multiclass SVM",
    y_test,
    baseline_test_predictions,
    label_mapping,
)


Validation results - baseline multiclass SVM
Accuracy: 0.6407
Balanced accuracy: 0.4618
Macro F1: 0.4723

Classification report:

              precision    recall  f1-score   support

         Low       0.00      0.00      0.00        10
      Medium       0.63      0.60      0.62       183
        High       0.64      0.74      0.68       258
    Critical       0.70      0.51      0.59        89

    accuracy                           0.64       540
   macro avg       0.49      0.46      0.47       540
weighted avg       0.63      0.64      0.63       540



,pred_Low,pred_Medium,pred_High,pred_Critical
actual_Low,0,7,3,0
actual_Medium,2,110,68,3
actual_High,0,51,191,16
actual_Critical,0,6,38,45


Test results - baseline multiclass SVM
Accuracy: 0.6241
Balanced accuracy: 0.4144
Macro F1: 0.4281

Classification report:

              precision    recall  f1-score   support

         Low       0.00      0.00      0.00         6
      Medium       0.68      0.52      0.59       192
        High       0.61      0.79      0.69       267
    Critical       0.59      0.35      0.44        75

    accuracy                           0.62       540
   macro avg       0.47      0.41      0.43       540
weighted avg       0.62      0.62      0.61       540



,pred_Low,pred_Medium,pred_High,pred_Critical
actual_Low,0,5,1,0
actual_Medium,1,100,88,3
actual_High,0,41,211,15
actual_Critical,0,2,47,26


## 2. Cost-sensitive multiclass SVM classifier

In [26]:
classes = np.array(sorted(label_mapping), dtype=int)
base_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)
class_weight_map = {label: float(weight) for label, weight in zip(classes, base_class_weights)}

severity_cost_multiplier = {
    label: 1.0 + (label / max(classes))
    for label in classes
}
weighted_class_map = {
    label: class_weight_map[label] * severity_cost_multiplier[label]
    for label in classes
}

pd.DataFrame(
    {
        "urgency_label": classes,
        "urgency": [label_mapping[label] for label in classes],
        "class_weight": [class_weight_map[label] for label in classes],
        "severity_multiplier": [severity_cost_multiplier[label] for label in classes],
        "effective_weight": [weighted_class_map[label] for label in classes],
    }
)


,urgency_label,urgency,class_weight,severity_multiplier,effective_weight
0,0,Low,11.250000,1.000000,11.250000
1,1,Medium,0.798479,1.333333,1.064639
2,2,High,0.494118,1.666667,0.823529
3,3,Critical,1.575000,2.000000,3.150000


In [27]:
cost_sensitive_candidates = [
    {
        "model_name": "weighted_svm_v1",
        "C": 0.75,
        "class_weight": weighted_class_map,
    },
    {
        "model_name": "weighted_svm_v2",
        "C": 1.0,
        "class_weight": weighted_class_map,
    },
    {
        "model_name": "weighted_svm_v3",
        "C": 1.5,
        "class_weight": weighted_class_map,
    },
]

cost_sensitive_results = []
cost_sensitive_models = {}

for candidate in cost_sensitive_candidates:
    candidate_pipeline = build_text_svm_pipeline(
        LinearSVC(
            C=candidate["C"],
            class_weight=candidate["class_weight"],
            random_state=RANDOM_SEED,
            dual="auto",
        )
    )
    candidate_pipeline.fit(X_train, y_train)

    validation_predictions = candidate_pipeline.predict(X_validation)
    validation_accuracy = accuracy_score(y_validation, validation_predictions)
    validation_balanced_accuracy = balanced_accuracy_score(y_validation, validation_predictions)
    validation_macro_f1 = f1_score(y_validation, validation_predictions, average="macro")

    cost_sensitive_models[candidate["model_name"]] = candidate_pipeline
    cost_sensitive_results.append(
        {
            "model_name": candidate["model_name"],
            "C": candidate["C"],
            "validation_accuracy": float(validation_accuracy),
            "validation_balanced_accuracy": float(validation_balanced_accuracy),
            "validation_macro_f1": float(validation_macro_f1),
        }
    )

cost_sensitive_results_df = pd.DataFrame(cost_sensitive_results).sort_values(
    by=["validation_balanced_accuracy", "validation_macro_f1", "validation_accuracy"],
    ascending=False,
).reset_index(drop=True)

best_cost_sensitive_model_name = cost_sensitive_results_df.iloc[0]["model_name"]
cost_sensitive_urgency_classifier = cost_sensitive_models[best_cost_sensitive_model_name]

cost_sensitive_results_df


,model_name,C,validation_accuracy,validation_balanced_accuracy,validation_macro_f1
0,weighted_svm_v1,0.75,0.622222,0.461265,0.462607
1,weighted_svm_v3,1.50,0.618519,0.458533,0.459531
2,weighted_svm_v2,1.00,0.618519,0.457090,0.459244


In [28]:
cost_validation_predictions = cost_sensitive_urgency_classifier.predict(X_validation)
cost_test_predictions = cost_sensitive_urgency_classifier.predict(X_test)

print(f"Selected cost-sensitive model: {best_cost_sensitive_model_name}")

print_multiclass_evaluation(
    "Validation results - cost-sensitive multiclass SVM",
    y_validation,
    cost_validation_predictions,
    label_mapping,
)

print_multiclass_evaluation(
    "Test results - cost-sensitive multiclass SVM",
    y_test,
    cost_test_predictions,
    label_mapping,
)


Selected cost-sensitive model: weighted_svm_v1
Validation results - cost-sensitive multiclass SVM
Accuracy: 0.6222
Balanced accuracy: 0.4613
Macro F1: 0.4626

Classification report:

              precision    recall  f1-score   support

         Low       0.00      0.00      0.00        10
      Medium       0.62      0.60      0.61       183
        High       0.64      0.68      0.66       258
    Critical       0.59      0.56      0.57        89

    accuracy                           0.62       540
   macro avg       0.46      0.46      0.46       540
weighted avg       0.62      0.62      0.62       540



,pred_Low,pred_Medium,pred_High,pred_Critical
actual_Low,0,8,2,0
actual_Medium,5,110,62,6
actual_High,1,52,176,29
actual_Critical,0,6,33,50


Test results - cost-sensitive multiclass SVM
Accuracy: 0.6204
Balanced accuracy: 0.4323
Macro F1: 0.4376

Classification report:

              precision    recall  f1-score   support

         Low       0.00      0.00      0.00         6
      Medium       0.68      0.56      0.62       192
        High       0.62      0.73      0.67       267
    Critical       0.49      0.44      0.46        75

    accuracy                           0.62       540
   macro avg       0.45      0.43      0.44       540
weighted avg       0.62      0.62      0.61       540



,pred_Low,pred_Medium,pred_High,pred_Critical
actual_Low,0,5,1,0
actual_Medium,2,108,77,5
actual_High,0,44,194,29
actual_Critical,0,2,40,33


## 3. Binary screening SVM classifier

In [29]:
positive_classes = {"High", "Critical"}

train_screen_df = train_df.copy()
validation_screen_df = validation_df.copy()
test_screen_df = test_df.copy()

for split_df in [train_screen_df, validation_screen_df, test_screen_df]:
    split_df["screening_label"] = split_df["urgency"].isin(positive_classes).astype(int)

X_train_screen = train_screen_df["feature_concatanation"]
y_train_screen = train_screen_df["screening_label"].astype(int)
X_validation_screen = validation_screen_df["feature_concatanation"]
y_validation_screen = validation_screen_df["screening_label"].astype(int)
X_test_screen = test_screen_df["feature_concatanation"]
y_test_screen = test_screen_df["screening_label"].astype(int)

screening_label_mapping = {0: "routine-review", 1: "immediate-attention"}

screening_classifier = build_text_svm_pipeline(
    CalibratedClassifierCV(
        estimator=LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            dual="auto",
        ),
        method="sigmoid",
        cv=3,
    )
)
screening_classifier.fit(X_train_screen, y_train_screen)

validation_probabilities = screening_classifier.predict_proba(X_validation_screen)[:, 1]
test_probabilities = screening_classifier.predict_proba(X_test_screen)[:, 1]

threshold_candidates = np.round(np.arange(0.20, 0.61, 0.01), 2)
threshold_rows = []

for threshold in threshold_candidates:
    validation_predictions = (validation_probabilities >= threshold).astype(int)
    threshold_rows.append(
        {
            "threshold": float(threshold),
            "validation_precision": float(precision_score(y_validation_screen, validation_predictions, zero_division=0)),
            "validation_recall": float(recall_score(y_validation_screen, validation_predictions, zero_division=0)),
            "validation_f1": float(f1_score(y_validation_screen, validation_predictions, zero_division=0)),
            "validation_accuracy": float(accuracy_score(y_validation_screen, validation_predictions)),
        }
    )

threshold_report_df = pd.DataFrame(threshold_rows)

eligible_thresholds = threshold_report_df[threshold_report_df["validation_recall"] >= TARGET_RECALL]
if not eligible_thresholds.empty:
    selected_threshold_row = eligible_thresholds.sort_values(
        by=["validation_precision", "validation_f1", "threshold"],
        ascending=[False, False, True],
    ).iloc[0]
else:
    selected_threshold_row = threshold_report_df.sort_values(
        by=["validation_recall", "validation_precision", "validation_f1", "threshold"],
        ascending=[False, False, False, True],
    ).iloc[0]

selected_threshold = float(selected_threshold_row["threshold"])
threshold_report_df.sort_values("threshold").reset_index(drop=True).head()


,threshold,validation_precision,validation_recall,validation_f1,validation_accuracy
0,0.20,0.665377,0.991354,0.796296,0.674074
1,0.21,0.666019,0.988473,0.795824,0.674074
2,0.22,0.667969,0.985591,0.796275,0.675926
3,0.23,0.668627,0.982709,0.795799,0.675926
4,0.24,0.673913,0.982709,0.799531,0.683333


In [30]:
print_binary_evaluation(
    "Validation results - binary screening SVM",
    y_validation_screen,
    validation_probabilities,
    selected_threshold,
    screening_label_mapping,
)

print_binary_evaluation(
    "Test results - binary screening SVM",
    y_test_screen,
    test_probabilities,
    selected_threshold,
    screening_label_mapping,
)


Validation results - binary screening SVM
Threshold: 0.37
Accuracy: 0.7333
Precision (immediate-attention): 0.7211
Recall (immediate-attention): 0.9539
F1 (immediate-attention): 0.8213
PR-AUC: 0.9083

Classification report:

                     precision    recall  f1-score   support

     routine-review       0.80      0.34      0.47       193
immediate-attention       0.72      0.95      0.82       347

           accuracy                           0.73       540
          macro avg       0.76      0.65      0.65       540
       weighted avg       0.75      0.73      0.70       540



,pred_routine-review,pred_immediate-attention
actual_routine-review,65,128
actual_immediate-attention,16,331


Test results - binary screening SVM
Threshold: 0.37
Accuracy: 0.7370
Precision (immediate-attention): 0.7128
Recall (immediate-attention): 0.9795
F1 (immediate-attention): 0.8251
PR-AUC: 0.9179

Classification report:

                     precision    recall  f1-score   support

     routine-review       0.90      0.32      0.47       198
immediate-attention       0.71      0.98      0.83       342

           accuracy                           0.74       540
          macro avg       0.81      0.65      0.65       540
       weighted avg       0.78      0.74      0.69       540



,pred_routine-review,pred_immediate-attention
actual_routine-review,63,135
actual_immediate-attention,7,335


In [31]:
with SCREENING_MODEL_PATH.open("wb") as model_file:
    pickle.dump(screening_classifier, model_file)

screening_metadata = {
    "train_path": str(TRAIN_PATH),
    "validation_path": str(VALIDATION_PATH),
    "test_path": str(TEST_PATH),
    "model_artifact_dir": str(SCREENING_ARTIFACT_DIR),
    "model_path": str(SCREENING_MODEL_PATH),
    "selected_threshold": selected_threshold,
    "target_recall": TARGET_RECALL,
    "label_mapping": {str(key): value for key, value in screening_label_mapping.items()},
    "random_seed": RANDOM_SEED,
    "feature_column": "feature_concatanation",
    "target_column": "screening_label",
    "target_name_column": "screening_target",
}

SCREENING_METADATA_PATH.write_text(json.dumps(screening_metadata, indent=2))

print(f"Saved binary screening SVM model to {SCREENING_MODEL_PATH}")
print(f"Saved binary screening metadata to {SCREENING_METADATA_PATH}")


Saved binary screening SVM model to /home/lakshan/ssp/IMS/train/artifacts/svm_urgency_binary_screening/model.pkl
Saved binary screening metadata to /home/lakshan/ssp/IMS/train/artifacts/svm_urgency_binary_screening/metadata.json
